# Keystroke Authentication - KeyRecs Dataset

Out-of-sample check for the method built on CMU. Alpha and target_far are
fixed at the values locked on CMU's validation sweep and not touched here.
The point is to see whether the method still works on a dataset it was
never tuned on, not to get the best possible number on this dataset.

In [30]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

from cmu_keyrecs_sharedfunctions import (
    build_scaled_manhattan_profile,
    scaled_manhattan_batch,
    threshold_at_far,
    find_balanced_accuracy_threshold,
)

# Locked on CMU's validation sweep — not re-tuned here.
LOCKED_ALPHA = 0.20
LOCKED_TARGET_FAR = 0.05

RANDOM_STATE = 42



## Load and inspect the data

KeyRecs, fixed-text condition: 99 participants, 2 sessions each, around
100 repetitions per session, 47 timing features.

In [31]:
keyrecs = pd.read_csv('fixed-text.csv')

feature_cols = [c for c in keyrecs.columns if c not in ('participant', 'session', 'repetition')]

print(keyrecs.shape)
print(keyrecs['participant'].nunique(), 'participants')
print(sorted(keyrecs['session'].unique()), 'sessions')
print(len(feature_cols), 'features')
keyrecs.head()

(19772, 50)
99 participants
[np.int64(1), np.int64(2)] sessions
47 features


,participant,session,repetition,DU.v.v,DD.v.p,DU.v.p,UD.v.p,UU.v.p,DU.p.p,DD.p.w,...,DU.r.k,UD.r.k,UU.r.k,DU.k.k.1,DD.k.b,DU.k.b,UD.k.b,UU.k.b,DU.b.b,total time
0,p001,1,1,0.129,1.917,1.804,2.046,1.933,0.113,0.719,...,1.289,1.509,1.423,0.086,0.450,0.352,0.536,0.438,0.098,10.101
1,p001,1,2,0.112,0.192,0.096,0.304,0.208,0.096,0.448,...,0.921,1.135,1.056,0.079,0.418,0.336,0.497,0.415,0.082,5.768
2,p001,1,3,0.088,0.253,0.182,0.341,0.270,0.071,1.536,...,1.066,1.271,1.194,0.077,0.448,0.352,0.525,0.429,0.096,7.877
3,p001,1,4,0.104,0.323,0.233,0.427,0.337,0.090,0.912,...,0.927,1.129,1.063,0.066,0.418,0.330,0.484,0.396,0.088,5.443
4,p001,1,5,0.107,0.188,0.100,0.295,0.207,0.088,0.858,...,1.237,1.423,1.344,0.079,0.457,0.369,0.536,0.448,0.088,5.284


## Checking session coverage

Confirming every participant has both sessions, and a similar number of
repetitions per session, before splitting anything by session below.

In [32]:

session_counts = keyrecs.groupby(['participant', 'session']).size().unstack()
print(session_counts.describe())
session_counts.isna().sum()


session           1           2
count     99.000000   99.000000
mean      99.777778   99.939394
std        0.693344    0.818388
min       95.000000   98.000000
25%      100.000000  100.000000
50%      100.000000  100.000000
75%      100.000000  100.000000
max      100.000000  107.000000


session
1    0
2    0
dtype: int64

## Same method as CMU, unchanged

Same profile, scoring, threshold, and adaptation logic as the CMU
notebook, imported from the shared file rather than rewritten here.
Session 1 is enrollment, split 60/40 into enroll and held-out data for
picking the threshold. Session 2 is the target the profile has to adapt
to. Alpha and target_far are the values locked on CMU, passed in as
defaults and not changed anywhere in this notebook.

In [33]:
def compute_threshold_and_curve_keyrecs(uid, data, feature_cols, enroll_frac=0.6, random_state=42):
    user_data = data[data['participant'] == uid]
    session1 = user_data[user_data['session'] == 1][feature_cols].to_numpy()

    n_enroll = int(len(session1) * enroll_frac)
    rs = np.random.default_rng(random_state)
    idx = rs.permutation(len(session1))
    enroll_idx, heldout_idx = idx[:n_enroll], idx[n_enroll:]

    enroll_data = session1[enroll_idx]
    heldout_genuine = session1[heldout_idx]

    profile_median, profile_mad = build_scaled_manhattan_profile(enroll_data)

    impostor_data = data[data['participant'] != uid][feature_cols].to_numpy()

    genuine_scores = scaled_manhattan_batch(heldout_genuine, profile_median, profile_mad)
    impostor_scores = scaled_manhattan_batch(impostor_data, profile_median, profile_mad)

    y_true = np.concatenate([np.zeros(len(genuine_scores)), np.ones(len(impostor_scores))])
    y_score = np.concatenate([genuine_scores, impostor_scores])

    strict_threshold = find_balanced_accuracy_threshold(y_true, y_score)

    return {
        'profile_median': profile_median,
        'profile_mad': profile_mad,
        'strict_threshold': strict_threshold,
        'y_true': y_true,
        'y_score': y_score,
    }


def adaptive_evaluate_user_keyrecs(uid, data, feature_cols, alpha=LOCKED_ALPHA, target_far=LOCKED_TARGET_FAR,
                                     enroll_frac=0.6, random_state=42):
    fit = compute_threshold_and_curve_keyrecs(uid, data, feature_cols, enroll_frac, random_state)
    profile_mean = fit['profile_median'].copy()
    profile_mad = fit['profile_mad']
    strict_threshold = fit['strict_threshold']
    adapt_threshold = threshold_at_far(fit['y_true'], fit['y_score'], target_far)

    user_data = data[data['participant'] == uid]
    session2 = user_data[user_data['session'] == 2].sort_values('repetition')

    outcomes = []
    n_adapted = 0
    for _, row in session2.iterrows():
        attempt = row[feature_cols].to_numpy(dtype=float)
        score = scaled_manhattan_batch(attempt.reshape(1, -1), profile_mean, profile_mad)[0]

        accepted = score <= strict_threshold
        outcomes.append({'score': score, 'accepted': accepted})

        if score <= adapt_threshold:
            profile_mean = (1 - alpha) * profile_mean + alpha * attempt
            n_adapted += 1

    genuine_scores = np.array([o['score'] for o in outcomes])
    frr = np.mean(genuine_scores > strict_threshold)

    impostor_data = data[data['participant'] != uid][feature_cols].to_numpy()
    impostor_scores = scaled_manhattan_batch(impostor_data, profile_mean, profile_mad)
    far = np.mean(impostor_scores <= strict_threshold)

    all_y_true = np.concatenate([np.zeros(len(genuine_scores)), np.ones(len(impostor_scores))])
    all_y_score = np.concatenate([genuine_scores, impostor_scores])
    fpr, tpr, _ = roc_curve(all_y_true, all_y_score)
    eer = fpr[np.nanargmin(np.abs(fpr - (1 - tpr)))]
    accuracy = 1 - (far + frr) / 2

    return {'far': far, 'frr': frr, 'eer': eer, 'accuracy': accuracy, 'n_adapted': n_adapted}


## Running this on every participant

No retuning happens here. This is the same function and parameters run once across all 99 participants.

In [34]:
all_participants = sorted(keyrecs['participant'].unique())

keyrecs_results = []
for uid in all_participants:
    r = adaptive_evaluate_user_keyrecs(uid, keyrecs, feature_cols)
    r['participant'] = uid
    keyrecs_results.append(r)

keyrecs_results_df = pd.DataFrame(keyrecs_results)
keyrecs_results_df.describe()

,far,frr,eer,accuracy,n_adapted
count,99.000000,99.000000,99.000000,99.000000,99.000000
mean,0.114338,0.237614,0.165153,0.824024,96.404040
std,0.099577,0.181736,0.101761,0.080667,10.178929
min,0.000307,0.010000,0.020000,0.499796,3.000000
25%,0.037962,0.120606,0.120000,0.775593,97.000000
50%,0.101012,0.180000,0.150000,0.838813,98.000000
75%,0.161276,0.300000,0.180909,0.874743,100.000000
max,0.486793,1.000000,0.960000,0.963409,105.000000


## Result

Averages across all 99 participants.

In [35]:
mean_far = keyrecs_results_df['far'].mean()
mean_frr = keyrecs_results_df['frr'].mean()
mean_eer = keyrecs_results_df['eer'].mean()
mean_accuracy = keyrecs_results_df['accuracy'].mean()

print(f"KeyRecs out-of-sample verification ({len(all_participants)} participants):")
print(f"  Mean FAR:      {mean_far:.3f}")
print(f"  Mean FRR:      {mean_frr:.3f}")
print(f"  Mean EER:      {mean_eer:.3f}")
print(f"  Mean Accuracy: {mean_accuracy:.3f}")

KeyRecs out-of-sample verification (99 participants):
  Mean FAR:      0.114
  Mean FRR:      0.238
  Mean EER:      0.165
  Mean Accuracy: 0.824


## Looking for the worst cases

Sorting by false reject rate to see which participants the method
struggles with most, and whether the profile ever adapted for them at
all.

In [36]:

keyrecs_results_df.sort_values('frr', ascending=False).head(10)

,far,frr,eer,accuracy,n_adapted,participant
3,0.000409,1.00,0.96,0.499796,3,p004
86,0.013283,0.89,0.27,0.548359,94,p087
76,0.017576,0.68,0.22,0.651212,99,p077
29,0.003372,0.67,0.18,0.663314,94,p030
9,0.044451,0.62,0.32,0.667774,97,p010
64,0.015991,0.61,0.24,0.687004,100,p065
24,0.127733,0.55,0.36,0.661133,98,p025
14,0.025393,0.54,0.25,0.717303,97,p015
40,0.001328,0.53,0.16,0.734336,93,p041
60,0.014102,0.50,0.14,0.742949,99,p061
